In [1]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

SEED = 1
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams["font.family"] = "Open Sans"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

"[info] base imports set"

Using device: cpu


'[info] base imports set'

In [2]:
GAMMA = 1.4
X_START, X_THROAT, X_LENGTH = 0.0, 0.5, 1.0
X_SHOCK_REF = 0.75  # reference/expected shock location
x_shock = 0.75  # initial single_loss_historyguess, updated after shock detection

T_STEADY = 5.0  # evaluate steady state at this pseudo-time

BD_DELTA = 1e-5  # mask width for inlet/exit/IC point selection
EVAL_PRED_POINTS = 2000  # number of points to evaluate the final solution

SINGLE_NET_MODEL_NAME = "single_net_model.pt"

# ── shock detection hyper-parameters ──────────────────────────────────────────
SHOCK_DET_POINTS = 2000

SHOCK_DET_ADAM_EPOCHS = 1000
SHOCK_DET_LBFGS_EPOCHS = 100
SHOCK_DET_ADAM_LR = 1e-3

SHOCK_DET_ADAM_RESAMPLE_EVERY = 500
SHOCK_DET_LBFGS_RESAMPLE_EVERY = 500

SHOCK_DET_PRINT_EVERY = 200

SHOCK_DET_GRAD_CLIP = 1.0
SHOCK_DET_BC_WEIGHT = 10.0
SHOCK_DET_FORMATION_TOL = 1e-1

# ── dual-network hyper-parameters ─────────────────────────────────────────────
DUAL_NET_POINTS = 1000
# DUAL_NET_TIME_SAMPLES = 200

DUAL_NET_ADAM_EPOCHS = 6000
DUAL_NET_LBFGS_EPOCHS = 1000
DUAL_NET_ADAM_LR = 1e-3
DUAL_NET_SHOCK_LR = 1e-4

DUAL_NET_ADAM_RESAMPLE_EVERY = 500
DUAL_NET_LBFGS_RESAMPLE_EVERY = 500

DUAL_NET_PRINT_EVERY = 500

DUAL_MAX_CONSECUTIVE_NAN = 5
DUAL_NET_GRAD_CLIP = 1.0
DUAL_NET_BC_WEIGHT = 10.0
DUAL_NET_INTERFACE_WEIGHT = 10.0
DUAL_NET_SHOCK_PRIOR_WEIGHT = 5.0

# REMESH_EVERY = 600  # re-mesh every this many epochs during dual-net training
# REMESH_SHIFT_TOL = 1e-3

"[info] hyper-parameters set"

'[info] hyper-parameters set'

In [3]:
from typing import overload


@overload
def get_area(x: float) -> float: ...
@overload
def get_area(x: np.ndarray) -> np.ndarray: ...
@overload
def get_area(x: torch.Tensor) -> torch.Tensor: ...
def get_area(x):
    return 1.0 + 2.2 * (x - X_THROAT) ** 2.0


@overload
def get_d_area(x: float) -> float: ...
@overload
def get_d_area(x: np.ndarray) -> np.ndarray: ...
@overload
def get_d_area(x: torch.Tensor) -> torch.Tensor: ...
def get_d_area(x):
    return 4.4 * (x - X_THROAT)


class AnalyticSolution:
    def __init__(self, n_points: int = EVAL_PRED_POINTS):
        self.x_shock = X_SHOCK_REF
        self.x = np.linspace(X_START, X_START + X_LENGTH, n_points)
        self._solve()

    def _solve(self):
        x, x_shock = self.x, self.x_shock
        A = get_area(x)
        M = np.empty_like(x)

        sub_mask = x <= X_THROAT
        pre_mask = (x > X_THROAT) & (x <= x_shock)
        post_mask = x > x_shock

        for i in np.where(sub_mask)[0]:
            M[i] = AnalyticSolution._bisection_for_mach_no(
                A[i].item(), supersonic=False
            )
        for i in np.where(pre_mask)[0]:
            M[i] = AnalyticSolution._bisection_for_mach_no(A[i].item(), supersonic=True)

        M1 = AnalyticSolution._bisection_for_mach_no(get_area(x_shock), supersonic=True)
        M2 = np.sqrt(
            (1.0 + (GAMMA - 1.0) / 2.0 * M1**2.0)
            / (GAMMA * M1**2.0 - (GAMMA - 1.0) / 2.0)
        )

        p02_by_p01 = (
            (((GAMMA + 1.0) * M1**2.0) / (2.0 + (GAMMA - 1.0) * M1**2.0))
            ** (GAMMA / (GAMMA - 1.0))
        ) * (
            ((GAMMA + 1.0) / (2.0 * GAMMA * M1**2.0 - GAMMA + 1.0))
            ** (1.0 / (GAMMA - 1.0))
        )

        a_star_2_by_a_star_1 = 1.0 / p02_by_p01

        for i in np.where(post_mask)[0]:
            a_eff = A[i] / a_star_2_by_a_star_1
            M[i] = AnalyticSolution._bisection_for_mach_no(a_eff, supersonic=False)

        T0 = np.ones_like(x)
        P0 = np.where(post_mask, p02_by_p01, 1.0)
        rho0 = P0 / T0

        T = T0 / (1.0 + (GAMMA - 1.0) / 2.0 * M**2.0)
        P = P0 * (1.0 + (GAMMA - 1.0) / 2.0 * M**2.0) ** (-GAMMA / (GAMMA - 1.0))
        rho = rho0 * (1.0 + (GAMMA - 1.0) / 2.0 * M**2.0) ** (-1.0 / (GAMMA - 1.0))

        self.M, self.P, self.T, self.rho = M, P, T, rho
        self.sub_mask, self.pre_mask, self.post_mask = sub_mask, pre_mask, post_mask
        self.M1, self.M2, self.p02_by_p01 = M1, M2, p02_by_p01

    def data(self, region: str):
        match region:
            case "subsonic":
                mask = self.sub_mask
            case "supersonic":
                mask = self.pre_mask
            case "post-shock":
                mask = self.post_mask
            case _:
                raise ValueError(f"Unknown region: {region}")

        return self.x[mask], (self.M[mask], self.P[mask], self.T[mask], self.rho[mask])

    @staticmethod
    def _bisection_for_mach_no(a: float, supersonic: bool) -> float:
        def func(M):
            t = 1.0 + (GAMMA - 1.0) / 2.0 * M**2.0
            expo = (GAMMA + 1.0) / (2.0 * (GAMMA - 1.0))
            return (1.0 / M) * (2.0 / (GAMMA + 1.0) * t) ** expo - a

        low, high = (1.0 + BD_DELTA, 10.0) if supersonic else (BD_DELTA, 1.0 - BD_DELTA)
        for _ in range(300):
            mid = (low + high) / 2
            if func(mid) * func(low) < 0:
                high = mid
            else:
                low = mid
        return (low + high) / 2


analytic_solution = AnalyticSolution()

M_INLET = AnalyticSolution._bisection_for_mach_no(get_area(X_START), supersonic=False)
RHO_INLET = (1.0 + (GAMMA - 1.0) / 2.0 * M_INLET**2.0) ** (-1.0 / (GAMMA - 1.0))
T_INLET = 1.0 / (1.0 + (GAMMA - 1.0) / 2.0 * M_INLET**2.0)
P_INLET, P_EXIT = RHO_INLET * T_INLET, analytic_solution.P[-1]
U_INLET = M_INLET * np.sqrt(GAMMA * T_INLET)

pd.DataFrame(
    {
        "M_INLET": [M_INLET],
        "RHO_INLET": [RHO_INLET],
        "T_INLET": [T_INLET],
        "P_INLET": [P_INLET],
        "U_INLET": [U_INLET],
        "P_EXIT": [P_EXIT],
    }
)

,M_INLET,RHO_INLET,T_INLET,P_INLET,U_INLET,P_EXIT
0,0.412857,0.919611,0.967034,0.889294,0.48038,0.828902


In [4]:
import torch.nn as nn

class UnsteadyNet(torch.nn.Module):

    """
    (x, t) -> (rho, u, P, T)
    """

    INPUT = 2
    OUTPUT = 4

    def __init__(self, hid_dim: int = 128, num_hidden: int = 3):
        super().__init__()

        layers = [nn.Linear(self.INPUT, hid_dim), nn.Tanh()]
        for _ in range(num_hidden):
            layers += [nn.Linear(hid_dim, hid_dim), nn.Tanh()]
        layers += [nn.Linear(hid_dim, self.OUTPUT)]

        self.net = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self):
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="tanh")
                nn.init.zeros_(m.bias)

    def forward(self, xt: torch.Tensor):
        out = self.net(xt)

        wrap = lambda x: nn.functional.softplus(x + 1.0)
        rho, u, P, T = map(lambda x: wrap(out[:, x:x + 1]), range(4))

        return torch.cat([rho, u, P, T], dim=1)


"[info] unsteady net defined"

'[info] unsteady net defined'

In [5]:
def sample_points(
    x_low: float,
    x_high: float,
    t_low: float = 0.0,
    t_high: float = T_STEADY,
    n: int = SHOCK_DET_POINTS,
    ic_frac: float = 0.05,
    bc_frac: float = 0.05,
) -> torch.Tensor:
    n_ic = max(1, int(n * ic_frac))
    n_bc = max(1, int(n * bc_frac))
    n_int = n - 2 * n_bc - n_ic

    # interior
    x_int = torch.rand(n_int, 1, device=device) * (x_high - x_low) + x_low
    t_int = torch.rand(n_int, 1, device=device) * (t_high - t_low) + t_low

    # inlet BC  (x = x_low, t ∈ [t_low, t_high])
    x_in = torch.full((n_bc, 1), x_low, device=device)
    t_in = torch.rand(n_bc, 1, device=device) * (t_high - t_low) + t_low

    # outlet BC (x = x_high, t ∈ [t_low, t_high])
    x_out = torch.full((n_bc, 1), x_high, device=device)
    t_out = torch.rand(n_bc, 1, device=device) * (t_high - t_low) + t_low

    # IC (t = 0, x ∈ [x_low, x_high])
    x_ic = torch.rand(n_ic, 1, device=device) * (x_high - x_low) + x_low
    t_ic = torch.zeros(n_ic, 1, device=device)

    xt = torch.cat(
        [
            torch.cat([x_int, t_int], dim=1),
            torch.cat([x_in, t_in], dim=1),
            torch.cat([x_out, t_out], dim=1),
            torch.cat([x_ic, t_ic], dim=1),
        ],
        dim=0,
    )

    idx = torch.randperm(xt.shape[0], device=device)
    return xt[idx].requires_grad_(True)


"[info] point sampler defined"

'[info] point sampler defined'

In [6]:
def _grad(f: torch.Tensor, xt: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    g = torch.autograd.grad(f.sum(), xt, create_graph=True)[0]
    return g[:, 0:1], g[:, 1:2]


class UnsteadyLoss:
    @staticmethod
    def pde(model: UnsteadyNet, xt: torch.Tensor):
        """
        - A * drho/dt - A * u * drho/dx - A * rho * du/dx - rho * u * dA/dx = 0
        - A * (rho * (du/dt + u * du/dx) + dP/dx) = 0
        - rho * A * (dT/dt + u * dT/dx) + (GAMMA - 1) * P * (A * du/dx + dA/dx * u) = 0
        - P - rho * T = 0
        """
        out = model(xt)
        rho, u, P, T = out[:, 0:1], out[:, 1:2], out[:, 2:3], out[:, 3:4]
        x = xt[:, 0:1]
        A, A_x = get_area(x), get_d_area(x)

        rho_x, rho_t = _grad(rho, xt)
        u_x, u_t = _grad(u, xt)
        P_x, _ = _grad(P, xt)
        T_x, T_t = _grad(T, xt)

        mass = A * rho_t + A * u * rho_x + A * rho * u_x + rho * u * A_x
        momentum = A * (rho * (u_t + u * u_x) + P_x)
        energy = rho * A * (T_t + u * T_x) + (GAMMA - 1.0) * P * (A * u_x + A_x * u)
        ideal = P - rho * T

        return torch.cat([mass, momentum, energy, ideal], dim=1)

    @staticmethod
    def bc(model: UnsteadyNet, xt: torch.Tensor):
        """
        MSE losses
        Inlet  (x≈0):  rho=1, u=U_INLET, P=1, T=1
        Outlet (x≈1):  P=P_EXIT
        IC     (t≈0):  rho=1, u=0, P=1, T=1
        """
        out = model(xt)
        rho, u, P, T = out[:, 0], out[:, 1], out[:, 2], out[:, 3]
        x, t = xt[:, 0], xt[:, 1]

        zero = torch.tensor(0.0, device=device)

        # inlet x≈0
        in_mask = x < BD_DELTA
        inlet_loss = (
            (
                (rho[in_mask] / RHO_INLET - 1.0).pow(2).mean()
                + (u[in_mask] - U_INLET).pow(2).mean()
                + (P[in_mask] / P_INLET - 1.0).pow(2).mean()
                + (T[in_mask] / T_INLET - 1.0).pow(2).mean()
            )
            if in_mask.any()
            else zero
        )

        # outlet x≈1
        out_mask = x > 1.0 - BD_DELTA
        outlet_loss = ((P[out_mask] / P_EXIT - 1.0).pow(2).mean()) if out_mask.any() else zero

        # IC at t=0
        ic_mask = t < BD_DELTA
        ic_loss = (
            (
                (rho[ic_mask] / RHO_INLET - 1.0).pow(2).mean()
                + (u[ic_mask] / U_INLET - 0.0).pow(2).mean()
                + (P[ic_mask] / P_INLET - 1.0).pow(2).mean()
                + (T[ic_mask] / T_INLET - 1.0).pow(2).mean()
            )
            if ic_mask.any()
            else zero
        )

        return inlet_loss + outlet_loss + ic_loss

    @staticmethod
    def total(
        model: UnsteadyNet, xt: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        pde_res = UnsteadyLoss.pde(model, xt)
        pde_loss = pde_res.pow(2).mean()
        bc_loss = UnsteadyLoss.bc(model, xt)
        loss = pde_loss + SHOCK_DET_BC_WEIGHT * bc_loss
        return loss, pde_loss, bc_loss


"[info] unsteady loss functions defined"

'[info] unsteady loss functions defined'

In [7]:
class ModelHandler:
    SAVE_DIR = "saved_models"

    @staticmethod
    def save_path(name: str) -> str:
        import os
        from datetime import datetime

        os.makedirs(ModelHandler.SAVE_DIR, exist_ok=True)
        ts = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        return os.path.join(ModelHandler.SAVE_DIR, f"{name}.{ts}.pt")

    @staticmethod
    def save(model: nn.Module, path: str):
        torch.save(model.state_dict(), path)
        print(f"  Saved: {path}")

    @staticmethod
    def snapshot(model: nn.Module) -> dict:
        from copy import deepcopy

        return deepcopy(model.state_dict())

    @staticmethod
    def update_best(model, loss_val, best_loss, best_snap):
        if loss_val < best_loss:
            return loss_val, ModelHandler.snapshot(model)
        return best_loss, best_snap


"[info] ModelHandler defined"

'[info] ModelHandler defined'

In [8]:
def single_net_train(model: UnsteadyNet):
    history, temp, best_loss, best_snap = [], {}, float("inf"), {}

    def mk_closure(optim: torch.optim.Optimizer, xt: torch.Tensor):

        def closure():
            optim.zero_grad()
            loss, pde_loss, bc_loss = UnsteadyLoss.total(model, xt)
            temp["nan"] = False
            if not torch.isfinite(loss):
                temp["nan"] = True
                temp["loss"] = float("nan")
                temp["pde_loss"] = float("nan")
                temp["bc_loss"] = float("nan")
                return loss

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), SHOCK_DET_GRAD_CLIP)
            temp["loss"] = loss.item()
            temp["pde_loss"] = pde_loss.item()
            temp["bc_loss"] = bc_loss.item()

            return loss

        return closure

    # --- ADAM ---
    optim = torch.optim.Adam(model.parameters(), lr=SHOCK_DET_ADAM_LR)
    xt = sample_points(X_START, X_START + X_LENGTH, n=SHOCK_DET_POINTS)
    closure = mk_closure(optim, xt)

    print(f"[info] ADAM on {SHOCK_DET_ADAM_EPOCHS} epochs")
    for epoch in range(SHOCK_DET_ADAM_EPOCHS):
        if epoch % SHOCK_DET_ADAM_RESAMPLE_EVERY == 0:
            xt = sample_points(X_START, X_START + X_LENGTH, n=SHOCK_DET_POINTS)

        loss = closure()
        optim.step()
        history.append((loss.item(), temp["pde_loss"], temp["bc_loss"]))
        best_loss, best_snap = ModelHandler.update_best(
            model, loss.item(), best_loss, best_snap
        )

        if (epoch + 1) % SHOCK_DET_PRINT_EVERY == 0:
            print(
                f"  epoch {epoch+1:>{len(str(SHOCK_DET_ADAM_EPOCHS))}}/{SHOCK_DET_ADAM_EPOCHS}  "
                f"pde={temp['pde_loss']:.4e}  bc={temp['bc_loss']:.4e}  "
                f"total={temp['loss']:.4e}"
            )

    model.load_state_dict(best_snap)

    # --- LBFGS ---
    epoch, consecutive_nans, temp = 0, 0, {}
    print(f"[info] L-BFGS on {SHOCK_DET_LBFGS_EPOCHS} epochs")
    while epoch < SHOCK_DET_LBFGS_EPOCHS:
        lbfgs = torch.optim.LBFGS(
            model.parameters(),
            line_search_fn="strong_wolfe",
            lr=0.5 ** (consecutive_nans + 1),
        )

        xt = sample_points(X_START, X_START + X_LENGTH, n=SHOCK_DET_POINTS)
        closure = mk_closure(lbfgs, xt)
        chunk_end = min(epoch + SHOCK_DET_LBFGS_RESAMPLE_EVERY, SHOCK_DET_LBFGS_EPOCHS)

        for _ in range(chunk_end - epoch):
            lbfgs.step(closure)
            epoch += 1
            if temp.get("nan", False):
                model.load_state_dict(best_snap)
                consecutive_nans += 1
                print(f"  epoch {epoch}: non-finite loss (attempt {consecutive_nans})")
                break

            consecutive_nans = 0
            best_loss, best_snap = ModelHandler.update_best(
                model, temp["loss"], best_loss, best_snap
            )
            history.append((temp["pde_loss"], temp["bc_loss"], temp["loss"]))
            if (epoch + 1) % max(1, SHOCK_DET_PRINT_EVERY // 5) == 0:
                print(
                    f"  L-BFGS {epoch+1:>{len(str(SHOCK_DET_LBFGS_EPOCHS))}}/{SHOCK_DET_LBFGS_EPOCHS}  "
                    f"pde={temp['pde_loss']:.4e}  bc={temp['bc_loss']:.4e}  "
                    f"total={temp['loss']:.4e}"
                )

        if consecutive_nans >= DUAL_MAX_CONSECUTIVE_NAN:
            print(
                f"  L-BFGS phase aborted at epoch {epoch}: {DUAL_MAX_CONSECUTIVE_NAN} consecutive non-finite steps."
            )
            break

    model.load_state_dict(best_snap)
    path = ModelHandler.save_path(SINGLE_NET_MODEL_NAME)
    ModelHandler.save(model, path)
    return history, path

single_net = UnsteadyNet().to(device)
single_net_history, single_net_path = single_net_train(single_net)

[info] ADAM on 1000 epochs
  epoch  200/1000  pde=2.9856e-02  bc=5.2913e-03  total=8.2769e-02
  epoch  400/1000  pde=2.0863e-02  bc=3.9875e-03  total=6.0738e-02
  epoch  600/1000  pde=2.1962e-02  bc=7.2857e-03  total=9.4819e-02
  epoch  800/1000  pde=1.8842e-02  bc=2.4355e-03  total=4.3198e-02
  epoch 1000/1000  pde=1.7720e-02  bc=2.4309e-03  total=4.2028e-02
[info] L-BFGS on 100 epochs
  L-BFGS  40/100  pde=8.4922e-03  bc=4.6984e-05  total=8.9621e-03
  L-BFGS  80/100  pde=2.2886e-03  bc=6.2782e-06  total=2.3513e-03
  Saved: saved_models/single_net_model.pt.2026-09-12_01-10-11.pt


In [16]:
def save_history(history, path):
    df = pd.DataFrame(
        history, columns=["pde_loss", "bc_loss", "total_loss"]
    )

    csv_path = path.replace(".pt", ".csv")
    df.to_csv(csv_path, index=False)
    print(f"  Saved history: {csv_path}")

save_history(single_net_history, "single_net_history.csv")

  Saved history: single_net_history.csv
